# Turn 2 — backfill `parsed` and `score` on existing pickles

Temporary notebook parsers. AIT will own `parse.py` in Turn 3. `evaluate()` is not changed.

Writes two keys onto each successful pickle:

- `parsed`: list of response units, or `None` if parse failed
  - DAT: nouns (no empty pads, cap 10)
  - AUT: uses (list, not a joined string)
  - CWT: `[story]` (one-element list)
- `score`: `float` from `ait.evaluate()`, or `None`

`parsed is None` means the raw text was unusable. `parsed` is a list and `score is None` can still be a valid collect (DAT with fewer than 7 unique words, empty AUT uses after SemDis filters, etc.).

Run order: **DAT → AUT → CWT**. First `evaluate()` call per test downloads embeddings. CWT (BERT) is the slow one. `overwrite=False` is resumable.

## 0. Imports

In [1]:
from __future__ import annotations

import pickle
import re
from pathlib import Path

import automated_intelligence_tests as ait
import glove_word_embeddings as gwe
from tqdm import tqdm

print("ait", getattr(ait, "__version__", "?"))

ait 0.1.7


## 1. DAT — parse and score

In [ ]:
def parse_dat(raw):
    """Raw DAT text -> list of cleaned nouns, or None.

    Same split + gwe.pre.clean_word as getting_started.parse_dat,
    but does not pad with empty strings. Cap 10. None if nothing usable.
    """
    text = str(raw or "").strip().strip("\"'")
    if not text:
        return None
    nouns = []
    for tok in re.split(r"[,;\n\r]+", text):
        tok = re.sub(r"^\s*[\d\.\)\-]+\s*", "", tok)
        word = gwe.pre.clean_word(tok)
        if word and word not in nouns:
            nouns.append(word)
        if len(nouns) >= 10:
            break
    return nouns or None


def score_dat(parsed, stim=None):
    """parsed list -> DAT score (mean pairwise distance x 100) or None.

    Calls ait.evaluate("dat", list) unchanged. DAT needs no cue.
    Score is None when fewer than 7 unique valid words.
    """
    if not parsed:
        return None
    try:
        result = ait.evaluate("dat", list(parsed))
    except Exception as e:
        print("score_dat failed:", e)
        return None
    if not result:
        return None
    return result.get("score")

## 2. AUT — parse and score

In [ ]:
def parse_aut(raw):
    """Raw AUT text -> list of cleaned uses, or None.

    Same line split / numbering strip / clean_word as getting_started.parse_aut,
    but returns the list instead of a comma-joined string.
    """
    text = re.sub(r"<br\s*/?>", "\n", str(raw or ""), flags=re.I)
    uses = []
    for line in re.split(r"[\n\r,;]+", text):
        line = re.sub(r"^\s*[\d\.\)\-\*]+\s*", "", line).strip()
        low = line.lower()
        if low.startswith(("here are", "here is")) or "creative uses" in low:
            continue
        toks = [t for t in (gwe.pre.clean_word(t) for t in line.split()) if t]
        if toks:
            uses.append(" ".join(toks))
    return uses or None


def score_aut(parsed, stim=None):
    """parsed uses + stim cue -> AUT SemDis score or None.

    Wraps into ait.evaluate("aut", {"cue", "responses"}) unchanged.
    Cue comes from pickle["kwargs"]["cue"]. Never invent a cue.
    """
    if not parsed:
        return None
    stim = stim or {}
    cue = stim.get("cue") or ""
    if isinstance(cue, (list, tuple)):
        cue = " ".join(str(x) for x in cue if str(x).strip())
    try:
        result = ait.evaluate("aut", {"cue": cue, "responses": list(parsed)})
    except Exception as e:
        print("score_aut failed:", e)
        return None
    if not result:
        return None
    return result.get("score")

## 3. CWT — parse and score

In [ ]:
def parse_cwt(raw):
    """Raw CWT text -> [story] (one-element list), or None.

    Same heading / Title strip as getting_started.parse_cwt.
    Wrapped in a list so parsed is list|None for every test.
    """
    text = re.sub(r"^#+\s*.*$", "", str(raw or ""), flags=re.M)
    text = re.sub(r"^\s*Title:.*$", "", text, flags=re.M | re.I)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return [text] if text else None


def score_cwt(parsed, stim=None):
    """parsed [story] + stim cue -> CWT BERT DSI score or None.

    Unwraps parsed[0]. Cue is kwargs["cue"] as stored (list or string).
    DSI scores the story; cue is passed through but does not change the number.
    """
    if not parsed:
        return None
    stim = stim or {}
    cue = stim.get("cue") or []
    if isinstance(cue, str):
        cue = [w for w in re.split(r"[,\s]+", cue) if w]
    try:
        result = ait.evaluate("cwt", {"cue": list(cue), "story": parsed[0]})
    except Exception as e:
        print("score_cwt failed:", e)
        return None
    if not result:
        return None
    return result.get("score")

## 4. Backfill walker

Resumable. `overwrite=False` skips a pickle once **both** `parsed` and `score` keys exist, even if the values are `None`. That way DAT < 7 and parse-failures are not rescored forever. Writes each file immediately so a crash keeps finished work.

In [ ]:
PARSE_FN = {"dat": parse_dat, "aut": parse_aut, "cwt": parse_cwt}
SCORE_FN = {"dat": score_dat, "aut": score_aut, "cwt": score_cwt}


def _task_dir(task, data_root="data"):
    task = task.strip().lower()
    for root in (Path(data_root) / task, Path(task)):
        if root.is_dir():
            return root
    raise FileNotFoundError(f"no pickle folder for {task!r}")


def _write_pickle(path, row):
    tmp = path.with_name(path.name + ".tmp")
    with tmp.open("wb") as f:
        pickle.dump(row, f, protocol=pickle.HIGHEST_PROTOCOL)
    tmp.replace(path)


def backfill_task(task, data_root="data", overwrite=False, limit=None, dry_run=False):
    """Walk data/<task>/**/*.pickle and write parsed + score in place.

    overwrite=False skips a file once both keys exist (even if they are None),
    so DAT <7 and parse-failures are not rescored forever.
    """
    task = task.strip().lower()
    parse_fn = PARSE_FN[task]
    score_fn = SCORE_FN[task]
    files = sorted(p for p in _task_dir(task, data_root).rglob("*.pickle") if p.is_file())
    if limit is not None:
        files = files[:limit]
    counts = dict(seen=0, wrote=0, skipped=0, untouched=0,
                  parse_none=0, score_none=0, errors=0)
    for p in tqdm(files, desc=f"backfill {task}"):
        try:
            with p.open("rb") as f:
                row = pickle.load(f)
        except Exception as e:
            print("SKIP", p, e)
            counts["errors"] += 1
            continue
        if row.get("error") or not row.get("raw"):
            counts["untouched"] += 1
            continue
        counts["seen"] += 1
        if not overwrite and "parsed" in row and "score" in row:
            counts["skipped"] += 1
            continue
        stim = row.get("kwargs") or {}
        parsed = parse_fn(row.get("raw"))
        score = score_fn(parsed, stim)
        row["parsed"] = parsed
        row["score"] = score
        if parsed is None:
            counts["parse_none"] += 1
        if score is None:
            counts["score_none"] += 1
        if not dry_run:
            _write_pickle(p, row)
            counts["wrote"] += 1
    print(task, counts, "[dry-run]" if dry_run else "")
    return counts


## 5. Smoke test — parse/score one pickle per test, no write

In [ ]:
def _first_good_pickle(task, data_root="data"):
    for p in _task_dir(task, data_root).rglob("*.pickle"):
        try:
            with p.open("rb") as f:
                row = pickle.load(f)
        except Exception:
            continue
        if not row.get("error") and row.get("raw"):
            return p, row
    return None, None


for task, parse_fn, score_fn in (
    ("dat", parse_dat, score_dat),
    ("aut", parse_aut, score_aut),
    ("cwt", parse_cwt, score_cwt),
):
    try:
        p, row = _first_good_pickle(task)
    except FileNotFoundError as e:
        print(task, e)
        continue
    if row is None:
        print(task, "no good pickle")
        continue
    parsed = parse_fn(row["raw"])
    score = score_fn(parsed, row.get("kwargs") or {})
    preview = parsed if not parsed or task != "cwt" else [parsed[0][:180] + "..."]
    print(f"\n{task}  {p}")
    print("  raw[:160]:", str(row["raw"])[:160].replace("\n", " / "))
    print("  parsed:", preview)
    print("  score:", score)

## 6. Run backfill

Uncomment one task at a time. Start with `limit=5` once, then drop `limit` and let it run.

```python
backfill_task("dat", limit=5)   # trial
backfill_task("dat")            # full DAT
backfill_task("aut")            # then AUT (5 embedding spaces)
backfill_task("cwt")            # then CWT (BERT, slowest)
```

In [ ]:
# Trial first — 5 pickles, real write. Comment out after it looks right.
backfill_task("dat", limit=5)

# Full runs (uncomment one at a time; overwrite=False resumes):
# backfill_task("dat")
# backfill_task("aut")
# backfill_task("cwt")